In [41]:
!pip install tiktoken -q

In [42]:
import sys, json
from pathlib import Path
from collections import defaultdict

sys.path.insert(0, str(Path("..").resolve()))

import tiktoken
import pandas as pd
from src.utils.prompts import PROMPT2CHARTCLASS

In [43]:
PREDICTIONS_DIR = Path("../outputs/predictions")

MODELS = {
    "GPT-4o": {
        "dir":          "gpt-4o",
        "image_tokens": 765,
        "input_price":  2.50,   # $/1M tokens
        "output_price": 10.00,
        "has_schema":   True,   # response_format json_schema counted as input tokens by OpenAI
    },
    "GPT-4o-mini": {
        "dir":          "gpt-4o-mini",
        "image_tokens": 765,
        "input_price":  0.15,
        "output_price": 0.60,
        "has_schema":   True,
    },
    "Gemini 2.5 Flash": {
        "dir":          "gemini-2.5-flash",
        "image_tokens": 258,
        "input_price":  3.50,   # thinking tokens billed at higher rate (thinking on by default)
        "output_price": 2.50,
        "has_schema":   False,
    },
    "Claude Sonnet 4.5": {
        "dir":          "claude-sonnet-4-5",
        "image_tokens": 786,
        "input_price":  3.00,
        "output_price": 15.00,
        "has_schema":   False,
    },
}

**Token counting note:** `cl100k_base` is the exact encoding for GPT-4o and GPT-4o-mini.
For Gemini 2.5 Flash (SentencePiece) and Claude Sonnet 4.5 (BPE) it is an approximation;
actual token counts may differ by ~5–15% depending on content.

In [44]:
enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

# Pre-compute once — prompt text is identical for all files of the same chart type
PROMPT_TOKENS = {
    chart_type: count_tokens(prompt)
    for chart_type, prompt in PROMPT2CHARTCLASS.items()
}

# Pre-compute JSON schema token counts (sent as response_format by OpenAI)
from src.utils.schema_json import SCHEMA2CHARTCLASS
SCHEMA_TOKENS = {
    chart_type: count_tokens(json.dumps(schema["schema"]))
    for chart_type, schema in SCHEMA2CHARTCLASS.items()
}

print("Prompt + schema token counts by chart type:")
print(f"  {'type':12s}  {'prompt':>7}  {'schema':>7}  {'total':>7}")
for ct in sorted(PROMPT_TOKENS):
    p = PROMPT_TOKENS[ct]
    s = SCHEMA_TOKENS.get(ct, 0)
    print(f"  {ct:12s}  {p:7d}  {s:7d}  {p+s:7d}")

Prompt + schema token counts by chart type:
  type           prompt   schema    total
  area              292      236      528
  bar               292      236      528
  box               366      357      723
  bubble            339      275      614
  errorpoint        329      307      636
  heatmap           216      216      432
  histogram         292      236      528
  line              292      236      528
  pie               175      123      298
  radar             171      123      294
  scatter           236      188      424


In [45]:
DATASET_SIZE = 1450  # official benchmark size

records = []
# per_class[chart_type][model_name] = list of output token counts
per_class = defaultdict(lambda: defaultdict(list))

for model_name, cfg in MODELS.items():
    model_dir = PREDICTIONS_DIR / cfg["dir"]
    total_input = total_output = n = 0

    for json_path in sorted(model_dir.rglob("*.json")):
        chart_type = json_path.parent.name  # path: .../{model}/{source}/{chart_type}/file.json
        prompt_tok  = PROMPT_TOKENS.get(chart_type, 0)
        schema_tok  = SCHEMA_TOKENS.get(chart_type, 0) if cfg["has_schema"] else 0
        input_tok   = prompt_tok + schema_tok + cfg["image_tokens"]
        output_tok  = count_tokens(json_path.read_text())  # raw file incl. indentation

        total_input  += input_tok
        total_output += output_tok
        n += 1
        per_class[chart_type][model_name].append(output_tok)

    avg_in  = total_input  / n
    avg_out = total_output / n
    total_cost = (
        (avg_in  / 1e6) * cfg["input_price"] +
        (avg_out / 1e6) * cfg["output_price"]
    ) * DATASET_SIZE

    records.append({
        "Model":             model_name,
        "Avg Input Tokens":  round(avg_in,  1),
        "Avg Output Tokens": round(avg_out, 1),
        "N Charts":          DATASET_SIZE,
        "In. Price ($/1M)":  cfg["input_price"],
        "Out. Price ($/1M)": cfg["output_price"],
        "Total Cost ($)":    round(total_cost, 4),
    })
    print(f"{model_name}: {n} actual charts | total cost (norm. {DATASET_SIZE}) ${total_cost:.2f}")

df = pd.DataFrame(records)

GPT-4o: 1603 actual charts | total cost (norm. 1450) $11.05
GPT-4o-mini: 1602 actual charts | total cost (norm. 1450) $0.59
Gemini 2.5 Flash: 1614 actual charts | total cost (norm. 1450) $4.60
Claude Sonnet 4.5: 1601 actual charts | total cost (norm. 1450) $15.04


In [46]:
display(
    df.style
    .format({
        "Avg Input Tokens":  "{:.0f}",
        "Avg Output Tokens": "{:.0f}",
        "N Charts":          "{:,}",
        "In. Price ($/1M)":  "${:.2f}",
        "Out. Price ($/1M)": "${:.2f}",
        "Total Cost ($)":    "${:.2f}",
    })
    .hide(axis="index")
    .set_caption("Estimated inference cost (tiktoken cl100k_base, normalized to 1,450 charts)")
)

Model,Avg Input Tokens,Avg Output Tokens,N Charts,In. Price ($/1M),Out. Price ($/1M),Total Cost ($)
GPT-4o,1215,458,"1,450",$2.50,$10.00,$11.05
GPT-4o-mini,1215,371,"1,450",$0.15,$0.60,$0.59
Gemini 2.5 Flash,501,568,"1,450",$3.50,$2.50,$4.60
Claude Sonnet 4.5,1029,486,"1,450",$3.00,$15.00,$15.04


In [47]:
# Average output tokens per chart type, per model
model_names = list(MODELS.keys())
chart_types = sorted(per_class.keys())

rows_class = []
for ct in chart_types:
    row = {"Chart Type": ct}
    for model_name in model_names:
        vals = per_class[ct][model_name]
        row[model_name] = round(sum(vals) / len(vals), 1) if vals else None
    rows_class.append(row)

df_class = pd.DataFrame(rows_class).set_index("Chart Type")

display(
    df_class.style
    .format("{:.1f}", na_rep="—")
    .set_caption("Avg output tokens per chart type per model")
)

,GPT-4o,GPT-4o-mini,Gemini 2.5 Flash,Claude Sonnet 4.5
Chart Type,,,,
bar,386.7,277.5,412.9,396.1
box,489.5,402.8,543.5,477.5
bubble,504.0,395.7,729.9,632.0
errorpoint,561.9,402.7,640.8,541.7
heatmap,725.5,656.5,760.4,759.0
heatmap_raw,233.7,232.7,241.0,186.5
histogram,404.5,355.5,786.0,583.9
line,519.1,379.5,640.9,551.7
pie,205.0,202.5,212.9,207.5


In [48]:
rows = []
for _, r in df.iterrows():
    rows.append(
        f"  {r['Model']} & {r['Avg Input Tokens']:.0f} & "
        f"\\${r['In. Price ($/1M)']:.2f} & "
        f"{r['Avg Output Tokens']:.0f} & "
        f"\\${r['Out. Price ($/1M)']:.2f} & "
        f"\\${r['Total Cost ($)']:.2f} \\\\"
    )

latex = "\n".join([
    r"\begin{table}[h]",
    r"\centering",
    r"\resizebox{\columnwidth}{!}{%",
    r"\begin{tabular}{lrrrrr}",
    r"\hline",
    r"  & \multicolumn{2}{c}{\textbf{Input}} & \multicolumn{2}{c}{\textbf{Output}} & \\",
    r"  \cline{2-3}\cline{4-5}",
    r"  Model & Avg Tok. & \$/1M & Avg Tok. & \$/1M & Total Cost (\$) \\",
    r"\hline",
    *rows,
    r"\hline",
    r"\end{tabular}}",
    f"\\caption{{Comparative analysis of inference costs and average token consumption across commercial models (N={DATASET_SIZE} charts).}}",
    r"\label{tab:model_costs}",
    r"\end{table}",
])

print(latex)

\begin{table}[h]
\centering
\resizebox{\columnwidth}{!}{%
\begin{tabular}{lrrrrr}
\hline
  & \multicolumn{2}{c}{\textbf{Input}} & \multicolumn{2}{c}{\textbf{Output}} & \\
  \cline{2-3}\cline{4-5}
  Model & Avg Tok. & \$/1M & Avg Tok. & \$/1M & Total Cost (\$) \\
\hline
  GPT-4o & 1215 & \$2.50 & 458 & \$10.00 & \$11.05 \\
  GPT-4o-mini & 1215 & \$0.15 & 371 & \$0.60 & \$0.59 \\
  Gemini 2.5 Flash & 501 & \$3.50 & 568 & \$2.50 & \$4.60 \\
  Claude Sonnet 4.5 & 1029 & \$3.00 & 486 & \$15.00 & \$15.04 \\
\hline
\end{tabular}}
\caption{Comparative analysis of inference costs and average token consumption across commercial models (N=1450 charts).}
\label{tab:model_costs}
\end{table}
